# 🛢️ Modelo LSTM — Predicción de Días Restantes por Pozo
**Fuente:** Dataset Tratado

In [ ]:
import warnings
import numpy as np
import pandas as pd
from flask import Flask, jsonify
from flask_cors import CORS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential

# Inicializamos la aplicación Web Flask y habilitamos CORS para el Dashboard
app = Flask(__name__)
CORS(app)

# Variable global que actuará como base de datos en la memoria RAM
DATASET_EN_MEMORIA = []

def ejecutar_pipeline_analitico():
    global DATASET_EN_MEMORIA

    # ----------------------------------------------------------------
    # 1. CARGA DE DATOS (Tu entrada desde el repositorio/AWS)
    # ----------------------------------------------------------------
    URL_SHEETS = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQ_rlq6VhoU4fgWduJN1tTabZXLORDQpjYOxahQIBNiFHC6KQ2Kqi8hG4NvIIlRSnfEzr-yKm9K7EAa/pub?gid=1809707117&single=true&output=csv"

    print('📦 [Fase 1/5] Cargando matriz física de fluidos desde AWS/Data Lake...')
    df_base = pd.read_csv(URL_SHEETS)
    df_base['Date'] = pd.to_datetime(df_base['Date'])
    df_base = df_base.sort_values(by=['Well_ID', 'Date']).reset_index(drop=True)

    # ----------------------------------------------------------------
    # 2. CALCULAR TARGET (Dias_Restantes)
    # ----------------------------------------------------------------
    print('⏳ [Fase 2/5] Calculando target dinámico de declinación (RUL)...')
    df_modelo = df_base[df_base['Well_Type'].str.lower() == 'producer'].copy()
    df_modelo['Dias_Restantes'] = np.nan
    pozos = df_modelo['Well_ID'].unique()

    for well in pozos:
        df_well = df_modelo[df_modelo['Well_ID'] == well].copy()
        indices_well = df_well.index

        df_well['Oil_7d_mean'] = df_well['Oil_rate_bbl_d'].rolling(window=7, min_periods=1).mean()
        df_well['Oil_prev_7d_mean'] = df_well['Oil_7d_mean'].shift(7)
        df_well['Oil_futuro_30d_mean'] = df_well['Oil_rate_bbl_d'].shift(-30).rolling(window=30, min_periods=1).mean()
        df_well['Water_7d_mean'] = df_well['Water_rate_bbl_d'].rolling(window=7, min_periods=1).mean()

        df_well['Colapso_Real'] = (
            (df_well['Oil_7d_mean'] < (df_well['Oil_prev_7d_mean'] * 0.75)) &
            (df_well['Oil_futuro_30d_mean'] < (df_well['Oil_prev_7d_mean'] * 0.80)) &
            (df_well['Water_7d_mean'] > 1500)
        )

        fechas_colapso = df_well[df_well['Colapso_Real'] == True]['Date'].tolist()
        fecha_limite = fechas_colapso[0] if fechas_colapso else df_well['Date'].max()
        df_modelo.loc[indices_well, 'Dias_Restantes'] = (fecha_limite - df_well['Date']).dt.days

    df_modelo['Dias_Restantes'] = df_modelo['Dias_Restantes'].apply(lambda x: max(0, x))

    # ----------------------------------------------------------------
    # 3. CREAR VENTANAS 3D PARA LA LSTM
    # ----------------------------------------------------------------
    print('📦 [Fase 3/5] Estructurando tensores temporales (Ventana de 30 días)...')
    features = ['Oil_rate_bbl_d', 'Water_rate_bbl_d', 'Gas_rate_mscf_d']
    tamano_ventana = 30

    scaler = MinMaxScaler()
    df_norm = df_modelo.copy()
    df_norm[features] = scaler.fit_transform(df_norm[features])

    X_list, y_list = [], []
    for well in pozos:
        df_well = df_norm[df_norm['Well_ID'] == well].sort_values(by='Date')
        X_well = df_well[features].values
        y_well = df_well['Dias_Restantes'].values

        if len(X_well) > tamano_ventana:
            for i in range(len(X_well) - tamano_ventana):
                X_list.append(X_well[i: i + tamano_ventana])
                y_list.append(y_well[i + tamano_ventana])

    X, y = np.array(X_list), np.array(y_list)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # ----------------------------------------------------------------
    # 4. ENTRENAMIENTO DE LA RED NEURONAL LSTM
    # ----------------------------------------------------------------
    print('🧠 [Fase 4/5] Inicializando y calibrando arquitectura LSTM...')
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse')

    model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=64, verbose=0)
    print('   ↳ Red LSTM entrenada satisfactoriamente en memoria.')

    # ----------------------------------------------------------------
    # 5. UNIFICACIÓN DE PREDICCIONES Y BLINDAJE DE DATOS
    # ----------------------------------------------------------------
    print('🛸 [Fase 5/5] Corriendo inferencias de campo completo y formateando salida...')
    data_exportar = []

    for well_id in pozos:
        df_pozo = df_norm[df_norm['Well_ID'] == well_id].sort_values(by='Date').reset_index(drop=True)
        if len(df_pozo) <= tamano_ventana:
            continue

        X_pozo = np.array([df_pozo[features].values[i: i + tamano_ventana] for i in range(len(df_pozo) - tamano_ventana)])
        predicciones = model.predict(X_pozo, verbose=0).flatten()

        df_recortado = df_modelo[df_modelo['Well_ID'] == well_id].iloc[tamano_ventana:].copy()
        df_recortado['Prediccion_LSTM_Dias'] = predicciones
        data_exportar.append(df_recortado)

    df_unificado = pd.concat(data_exportar, ignore_index=True)

    # Forzar la fecha a formato ISO String plano
    df_unificado['Date'] = df_unificado['Date'].dt.strftime('%Y-%m-%d')

    # Convertimos la matriz final a una estructura de diccionario nativo en la RAM
    DATASET_EN_MEMORIA = df_unificado.to_dict(orient='records')
    print(f'🎯 ¡Matriz analítica lista! {len(DATASET_EN_MEMORIA)} registros acoplados en RAM.')

# ----------------------------------------------------------------
# 6. ENLACE DE RED (Ruta de la API REST)
# ----------------------------------------------------------------
@app.route('/api/predicciones', methods=['GET'])
def enviar_datos_dashboard():
    # Si la base de datos está vacía al consultar, ejecutamos el pipeline UNA sola vez bajo demanda.
    global DATASET_EN_MEMORIA
    if not DATASET_EN_MEMORIA:
        print("⚡ Petición entrante detectada. Ejecutando pipeline LSTM...")
        ejecutar_pipeline_analitico()
    return jsonify(DATASET_EN_MEMORIA)

if __name__ == '__main__':
    # 🛠️ AJUSTE ESTRATÉGICO DEFINITIVO:
    # Levantar primero el link web. La red LSTM solo se entrenará cuando abras el navegador.
    print("\n🌐 Servidor Web Activo - Esperando peticiones del frontend...")
    print("🔗 LINK EN VIVO DE TU API: http://127.0.0.1:5000/api/predicciones")

    # Desactivamos Debug y Reloader para blindar el puerto 5000 contra reinicios infinitos
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)


🌐 Servidor Web Activo - Esperando peticiones del frontend...
🔗 LINK EN VIVO DE TU API: http://127.0.0.1:5000/api/predicciones
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.74:5000
Press CTRL+C to quit
